# Test Model on CIFAR-10 Batch Files

This notebook tests the EfficientNet-B3 model (`4th_model_98percent.pth`) on CIFAR-10 batch files (data_batch_1 through data_batch_5).

**Purpose:**
- Load the trained model
- Evaluate on all 50,000 CIFAR-10 training images from batch files
- Calculate overall accuracy and per-class metrics


In [10]:
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import importlib

import models
import datasets
importlib.reload(models)
importlib.reload(datasets)

from models import create_efficientnet_b3
from datasets import CIFAR10BatchDataset300, INV_LABELS_DICT

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")


Device: cuda


## Configuration


In [ ]:
model_path = "generated_data/fine_tune_results/fine_tuned_model.pth"
batch_size = 128
num_workers = 4
pin_memory = True
persistent_workers = True
prefetch_factor = 2

print(f"Model path: {model_path}")


Model path: generated_dataine_tune_resultsine_tuned_model.pth


## Load Model and Dataset


In [12]:
# Load CIFAR-10 batch dataset (data_batch_1 through data_batch_5)
test_dataset = CIFAR10BatchDataset300(batch_dir="./cifar-10-batches-py")

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=pin_memory,
    persistent_workers=persistent_workers,
    prefetch_factor=prefetch_factor
)

print(f"Test samples: {len(test_dataset)}")

# Load model
model = create_efficientnet_b3(num_classes=10).to(device)
model.load_state_dict(torch.load(model_path, map_location=device))
model.eval()

print("Model loaded successfully!")


Test samples: 50000


OSError: [Errno 22] Invalid argument: 'generated_data\x0cine_tune_results\x0cine_tuned_model.pth'

## Evaluate Model


In [ ]:
all_predictions = []
all_labels = []

print("Evaluating model on CIFAR-10 batch files...")
with torch.no_grad():
    for x, y in test_loader:
        x = x.to(device)
        logits = model(x)
        _, predicted = logits.max(1)
        all_predictions.extend(predicted.cpu().numpy())
        all_labels.extend(y.numpy())

all_predictions = np.array(all_predictions)
all_labels = np.array(all_labels)

# Calculate overall accuracy
accuracy = accuracy_score(all_labels, all_predictions)
print(f"\nOverall Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")


Evaluating model on CIFAR-10 batch files...

Overall Accuracy: 1.0000 (100.00%)


## Detailed Metrics


In [ ]:
# Get class names
class_names = [INV_LABELS_DICT[i] for i in range(10)]

# Classification report
print("\nClassification Report:")
print(classification_report(all_labels, all_predictions, target_names=class_names, digits=4))



Classification Report:
              precision    recall  f1-score   support

       truck     1.0000    1.0000    1.0000      5000
        deer     1.0000    1.0000    1.0000      5000
        bird     1.0000    1.0000    1.0000      5000
        frog     1.0000    1.0000    1.0000      5000
        ship     1.0000    1.0000    1.0000      5000
       horse     0.9998    1.0000    0.9999      5000
         cat     1.0000    1.0000    1.0000      5000
         dog     1.0000    0.9998    0.9999      5000
  automobile     1.0000    1.0000    1.0000      5000
    airplane     1.0000    1.0000    1.0000      5000

    accuracy                         1.0000     50000
   macro avg     1.0000    1.0000    1.0000     50000
weighted avg     1.0000    1.0000    1.0000     50000



In [ ]:
# Confusion matrix
cm = confusion_matrix(all_labels, all_predictions)
print("\nConfusion Matrix:")
print(cm)

# Per-class accuracy
print("\nPer-Class Accuracy:")
for i, class_name in enumerate(class_names):
    class_mask = all_labels == i
    if class_mask.sum() > 0:
        class_acc = (all_predictions[class_mask] == all_labels[class_mask]).mean()
        print(f"{class_name:12s}: {class_acc:.4f} ({class_acc*100:.2f}%)")



Confusion Matrix:
[[5000    0    0    0    0    0    0    0    0    0]
 [   0 5000    0    0    0    0    0    0    0    0]
 [   0    0 5000    0    0    0    0    0    0    0]
 [   0    0    0 5000    0    0    0    0    0    0]
 [   0    0    0    0 5000    0    0    0    0    0]
 [   0    0    0    0    0 5000    0    0    0    0]
 [   0    0    0    0    0    0 5000    0    0    0]
 [   0    0    0    0    0    1    0 4999    0    0]
 [   0    0    0    0    0    0    0    0 5000    0]
 [   0    0    0    0    0    0    0    0    0 5000]]

Per-Class Accuracy:
truck       : 1.0000 (100.00%)
deer        : 1.0000 (100.00%)
bird        : 1.0000 (100.00%)
frog        : 1.0000 (100.00%)
ship        : 1.0000 (100.00%)
horse       : 1.0000 (100.00%)
cat         : 1.0000 (100.00%)
dog         : 0.9998 (99.98%)
automobile  : 1.0000 (100.00%)
airplane    : 1.0000 (100.00%)
